In [ ]:
# 确认环境
import transformers
import trl
import torch
import re, json

print(f"Transformers: {transformers.__version__}")  # >= 4.51.0
print(f"TRL: {trl.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")

In [ ]:
# 加载数据集
from datasets import load_dataset, Dataset
import datasets

tool_ds = Dataset.load_from_disk("./tool_ds_10k").shuffle(seed=24).select(range(40))
chat_ds = Dataset.load_from_disk("./chat_ds_10k").shuffle(seed=24).select(range(40))

print(f"Selected samples: {len(tool_ds)}")

In [ ]:
# list of dict -> list of tools:{name, description, parameters} if role == 'assistant' and functions is not None
def extract_functions(functions_str)-> list:
    data = json.loads(functions_str)   # 先转成 list[dict]
    return [f["function"] for f in data]

# list of dict, each dict has keys: tool_name, tool_results if role is environment
def parse_tool_result(content)-> list:
    results = []

    for line in content.split("\n"):
        line = line.strip()
        if line:
            try:
                results.append(json.loads(line))
            except:
                results.append(line)  # fallback

    return results

def parse_tool_calls(text):

    calls = []
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    for line in lines:
        match = re.match(r"(\w+(?:\.\w+)*)\((.*)\)", line)
        if not match:
            continue

        func_name = match.group(1)
        args_str = match.group(2)

        args = {}
        for k, v in re.findall(r"(\w+)\s*=\s*('[^']*'|\"[^\"]*\"|\d+)", args_str):
            v = v.strip()

            if v.startswith(("'", '"')) and v.endswith(("'", '"')):
                v = v[1:-1]
            else:
                if v.isdigit():
                    v = int(v)

            args[k] = v

        calls.append({
            "name": func_name,
            "arguments": args
        })

    return calls

In [ ]:
# only keep too calls and system prompt with tools
def process_message(example):
    messages = example["messages"]
    new_messages = []
    for i in messages[:3]:
        # tool_calls
        if i['role'] == 'assistant' and i['function_calls']:
            function_calls = parse_tool_calls(i['function_calls'])
            new_messages.append({
                'role': i['role'],
                "content": f"<tool_calls>{json.dumps(function_calls, ensure_ascii=False)}</tool_calls>"
            })
        # system prompt with tools
        if i['role'] == 'system' and i['functions']:
            functions = extract_functions(i['functions'])
            new_messages.append({
                'role': 'system',
                'content':f"""You are a helpful function-calling AI assistant.
You are provided with function signatures within <tools></tools> XML tags.
When needed, call tools by outputting tool calls in <tool_calls></tool_calls> XML tags.
Do not include any other text when making tool calls.
<tools>
{json.dumps(functions, ensure_ascii=False)}
</tools>
                """
            })
        # user content
        if i['role'] == 'user' and i['content']:
            new_messages.append({
                'role': i['role'],
                'content': i['content']
            })
    return {'messages': new_messages}

In [ ]:
tool_ds = tool_ds.map(process_message, num_proc=4)
print(tool_ds[0]['messages'])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

sft_mdoel = "./qwen3-1.7b-sft-tool_call-chat"
base_model = "Qwen/Qwen3-1.7B-Base"

def load_lora_model(base_model, lora_path):
    tokenizer = AutoTokenizer.from_pretrained(lora_path)

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model,
        torch_dtype=torch.float16,
        device_map='auto',
    )

    model = PeftModel.from_pretrained(base_model, lora_path, is_trainable=True)
    return model, tokenizer

model, tokenizer = load_lora_model(base_model, sft_mdoel)

In [ ]:
def convert_chat_to_grpo(example):
    messages = example["messages"]

    last_assistant_idx = next(
        i for i in range(len(messages)-1, -1, -1)
        if messages[i]["role"] == "assistant"
    )
    
    if len(last_assistant_idx) == 0:
        return None   # 或 skip sample

    prompt_messages = messages[:last_assistant_idx]
    answer = messages[last_assistant_idx]["content"]

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        truncation=True, # 不会破坏 special token,不会截断在奇怪位置
        max_length=1024,
    )

    return {
        "prompt": prompt,
        "answer": answer,
    }


In [ ]:
def convert_tool_to_grpo(sample):
    messages = sample['messages']

    prompt_messages = messages[:len(messages)-1]
    answer = messages[-1]["content"]

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        truncation=True,
        max_length=1024,
    )
    return {
        "prompt": prompt,
        "answer": answer,
    }

In [ ]:
tool_ds = tool_ds.map(convert_tool_to_grpo, num_proc=4)

In [ ]:
print(tool_ds[0]['prompt'])

In [ ]:
chat_ds = chat_ds.map(convert_chat_to_grpo, num_proc=4)
print(chat_ds[0]['answer'])

In [ ]:
cols_to_remove = [c for c in tool_ds.features if c not in ['prompt', 'answer']]
if cols_to_remove:
    tool_ds = tool_ds.remove_columns(cols_to_remove)
cols_to_remove = [c for c in chat_ds.column_names if c not in ['prompt', 'answer']]
if cols_to_remove:
    chat_ds = chat_ds.remove_columns(cols_to_remove)

In [ ]:
dataset = datasets.concatenate_datasets([tool_ds, chat_ds])

In [ ]:
print(dataset)

In [ ]:
print(dataset[0])

In [ ]:
lengths = [len(tokenizer(x["prompt"]).input_ids) for x in dataset]

print("avg:", sum(lengths)/len(lengths))
print("max:", max(lengths))

In [ ]:
def filter_long(example):
    length = len(tokenizer(example["prompt"]).input_ids)
    return length < 1024

dataset = dataset.filter(filter_long)

In [ ]:
lengths = [len(tokenizer(x["prompt"]).input_ids) for x in dataset]

print("avg:", sum(lengths)/len(lengths))
print("max:", max(lengths))

# reward

In [ ]:
import json
import re
from typing import List, Dict

def compute_tool_call_reward(pred, gt):
    pred_calls = extract_tool_calls(pred)
    gt_calls = extract_tool_calls(gt)

    if not gt_calls:
        return float(len(pred_calls) == 0)

    gt_map = {c["name"]: c["arguments"] for c in gt_calls}

    score = 0.0
    total = len(gt_calls)

    for c in pred_calls:
        name = c.get("name")
        args = c.get("arguments")

        if name in gt_map:
            if args == gt_map[name]:
                score += 1.0
            elif str(args) == str(gt_map[name]):
                score += 0.7
            else:
                score += 0.2

    reward = score / total

    # format bonus
    if len(pred_calls) > 0:
        reward += 0.1

    return min(reward, 1.0)


def extract_tool_calls(text: str) -> List[Dict]:
    """提取 <tool_calls> 中的 JSON"""
    match = re.search(r'<tool_calls>(.*?)</tool_calls>', text, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group(1)) # JSON 标准只允许双引号：
    except:
        return []


def extract_tool_blocks(text: str) -> List[str]:
    """提取所有 <tool_calls> 块"""
    return re.findall(r'<tool_calls>.*?</tool_calls>', text, re.DOTALL)


In [ ]:
import re

def normalize(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def compute_chat_quality(completion: str, answer: str) -> float:
    comp = normalize(completion)
    ans = normalize(answer)

    # 1. exact match
    if comp == ans:
        return 1.0

    # 2. containment (弱匹配)
    if ans in comp or comp in ans:
        return 0.7

    # 3. token overlap (轻量版)
    comp_tokens = set(comp.split())
    ans_tokens = set(ans.split())

    if len(ans_tokens) == 0:
        return 0.0

    overlap = len(comp_tokens & ans_tokens) / len(ans_tokens)

    return min(overlap, 1.0)

In [ ]:
def reward_fn(completions:list[str], answer:list[str], **kwargs)->list:
    rewards = []
    for completion, ans_text in zip(completions, answer):

        is_tool_task = "<tool_calls>" in ans_text
        if is_tool_task:
            gt = ans_text
            reward = compute_tool_call_reward(completion, gt)
        else:
            reward = compute_chat_quality(completion, ans_text)
            if "<tool_calls>" in completion:
                reward -= 0.5

        reward = max(-1.0, min(1.0, reward))
        rewards.append(reward)
        
    return rewards

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./grpo-qwen3-1.7b",

    # GRPO 核心参数
    max_completion_length = 512,

    beta=0.015,                # KL 惩罚系数（较小，允许策略更多变化）

    # 训练超参数
    learning_rate=5e-6,
    num_generations=4,             # 每个 prompt 采样 G 个回复
    per_device_train_batch_size=1, # 每个 step 的 prompt 数, 总 batch = G * per_device_train_batch_size
    gradient_accumulation_steps=8, # 梯度累积
    
    max_steps=500,                 # 最多训练 500 步, 每 step 耗时：(per_device_train_batch_size × num_generations × max_len) / GPU吞吐

    # 优化设置
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",       # 8bit 优化器

    # 日志和保存
    logging_steps=5,               # 每步记录（GRPO 训练中每步都很重要）
    save_steps=100,
    save_total_limit=3,
    # 其他
    seed=42,
    report_to="none",
)

In [ ]:
# 创建 GRPOTrainer
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
)

print("=" * 60)
print("开始 GRPO 训练")
print(f"组大小 G: {grpo_config.num_generations}")
print(f"最大步数: {grpo_config.max_steps}")
print(f"学习率: {grpo_config.learning_rate}")
print(f"KL 系数: {grpo_config.beta}")
print("=" * 60)

In [ ]:
result = trainer.train()
print(f"\n训练完成! 总步数: {result.global_step}")

# 保存最终模型
trainer.save_model("./grpo-qwen3-1.7b/final")
tokenizer.save_pretrained("./grpo-qwen3-1.7b/final")

In [ ]:
import json
import matplotlib.pyplot as plt

# 提取训练日志
log_history = trainer.state.log_history

# 提取关键指标
steps = []
reward_means = []
reward_stds = []
response_lengths = []
kl_values = []

for entry in log_history:
    steps.append(entry.get("step", 0))

    # reward
    reward_key = next(
        (k for k in entry.keys() if "rewards" in k and "mean" in k),
        None
    )
    if reward_key:
        reward_means.append(entry[reward_key])

    std_key = next(
        (k for k in entry.keys() if "rewards" in k and "std" in k),
        None
    )
    if std_key:
        reward_stds.append(entry[std_key])

    # length
    if "completions/mean_length" in entry:
        response_lengths.append(entry["completions/mean_length"])

    # kl
    if "kl" in entry:
        kl_values.append(entry["kl"])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 奖励均值（模型是否在学会做对题？）
axes[0, 0].plot(steps[:len(reward_means)], reward_means, 'b-', linewidth=1.5)
axes[0, 0].set_title('Reward Mean (accuracy)', fontsize=12)
axes[0, 0].set_xlabel('Training Steps')
axes[0, 0].set_ylabel('Mean Reward')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim(0, 1)

# 2. 奖励标准差（组内差异是否在变化？）
if reward_stds:
    axes[0, 1].plot(steps[:len(reward_stds)], reward_stds, 'r-', linewidth=1.5)
    axes[0, 1].set_title('Reward Std (group diversity)', fontsize=12)
    axes[0, 1].set_xlabel('Training Steps')
    axes[0, 1].set_ylabel('Reward Std')
    axes[0, 1].grid(True, alpha=0.3)

# 3. 平均回复长度（模型是否在学会更详细地推理？）
if response_lengths:
    axes[1, 0].plot(range(len(response_lengths)), response_lengths,
                     'g-', linewidth=1.5)
    axes[1, 0].set_title('Response Length (reasoning depth)', fontsize=12)
    axes[1, 0].set_xlabel('Training Steps')
    axes[1, 0].set_ylabel('Avg. Tokens')
    axes[1, 0].grid(True, alpha=0.3)

# 4. KL 散度（策略是否偏离参考模型过远？）
if kl_values:
    axes[1, 1].plot(range(len(kl_values)), kl_values, 'm-', linewidth=1.5)
    axes[1, 1].set_title('KL Divergence', fontsize=12)
    axes[1, 1].set_xlabel('Training Steps')
    axes[1, 1].set_ylabel('KL')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./grpo_training_curves.png", dpi=150)
plt.show()

print("训练曲线已保存至 grpo_training_curves.png")